# 2차시: Multi-Agent (AutoGen) 실습

AutoGen 기반 멀티 에이전트 시스템 구축 실습 노트북입니다.

## 목차
7. Tool Use (도구 사용)
8. FunctionTool과 고급 도구 패턴
9. Human-in-the-Loop (사람 개입)
10. Termination 조건
11. 에이전트 상태 저장과 복원
12. 디버깅과 모니터링
13. 통합 실습: 코드 리뷰 에이전트 팀
14. 실전 팁과 베스트 프랙티스

## 환경 설정

AutoGen 패키지 설치 및 API 키 설정

In [ ]:
# AutoGen 설치 (최초 1회)
# !pip install -U "autogen-agentchat" "autogen-ext[openai]"

In [ ]:
import os



In [6]:
from autogen_ext.models.openai import OpenAIChatCompletionClient

model_client = OpenAIChatCompletionClient(model="gpt-4o")

---
## 7. Tool Use (도구 사용)

일반 Python 함수를 도구로 등록하여 Agent가 외부 기능을 호출할 수 있게 합니다.
- 함수의 **타입 힌트**와 **독스트링**이 자동으로 도구 스키마로 변환됩니다.
- `reflect_on_tool_use=True`로 설정하면 도구 결과를 자연어로 정리합니다.

### 7-1. 기본 도구 정의 및 Agent 등록

In [7]:
from autogen_agentchat.agents import AssistantAgent
from autogen_agentchat.ui import Console


# 도구 1: 날씨 조회 (시뮬레이션)
async def get_weather(city: str) -> str:
    """주어진 도시의 현재 날씨를 조회합니다."""
    weather_data = {
        "서울": "맑음, 22°C, 습도 45%",
        "부산": "흐림, 19°C, 습도 70%",
        "제주": "비, 17°C, 습도 85%",
    }
    return weather_data.get(city, f"{city}: 날씨 데이터 없음")


# 도구 2: 수학 계산
async def calculate(expression: str) -> str:
    """수학 표현식을 안전하게 계산합니다. 예: '234 * 567', '(10 + 20) / 3'"""
    allowed = set("0123456789+-*/.() ")
    if not all(c in allowed for c in expression):
        return "오류: 허용되지 않는 문자가 포함되어 있습니다."
    try:
        result = eval(expression)
        return f"계산 결과: {result}"
    except Exception as e:
        return f"계산 오류: {str(e)}"


# Agent에 도구 등록
tool_agent = AssistantAgent(
    name="tool_agent",
    model_client=model_client,
    tools=[get_weather, calculate],
    system_message="도구를 사용하여 사용자의 요청을 처리하세요. 한국어로 답변하세요.",
    reflect_on_tool_use=True,  # 도구 결과를 자연어로 정리
)

# 실행
await Console(tool_agent.run_stream(task="서울과 부산의 날씨를 알려주고, 234 * 567을 계산해줘"))

---------- TextMessage (user) ----------
서울과 부산의 날씨를 알려주고, 234 * 567을 계산해줘
---------- ToolCallRequestEvent (tool_agent) ----------
[FunctionCall(id='call_F9po5NZambxVce4TlgAEsbeg', arguments='{"city": "서울"}', name='get_weather'), FunctionCall(id='call_ruqLuxw5pwFBqIhWJ6fwO7by', arguments='{"city": "부산"}', name='get_weather'), FunctionCall(id='call_3KPzmtuoTqH4BbapMysjipuj', arguments='{"expression": "234 * 567"}', name='calculate')]
---------- ToolCallExecutionEvent (tool_agent) ----------
[FunctionExecutionResult(content='맑음, 22°C, 습도 45%', name='get_weather', call_id='call_F9po5NZambxVce4TlgAEsbeg', is_error=False), FunctionExecutionResult(content='흐림, 19°C, 습도 70%', name='get_weather', call_id='call_ruqLuxw5pwFBqIhWJ6fwO7by', is_error=False), FunctionExecutionResult(content='계산 결과: 132678', name='calculate', call_id='call_3KPzmtuoTqH4BbapMysjipuj', is_error=False)]
---------- TextMessage (tool_agent) ----------
서울의 날씨는 맑고 기온은 22°C이며, 습도는 45%입니다. 
부산의 날씨는 흐리고 기온은 19°C이며, 습도는 70%입니다. 

TaskResult(messages=[TextMessage(id='34002a10-fddf-465e-a9d7-259aabe7f103', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 1, 20, 419070, tzinfo=datetime.timezone.utc), content='서울과 부산의 날씨를 알려주고, 234 * 567을 계산해줘', type='TextMessage'), ToolCallRequestEvent(id='f89757c5-f4a8-4d4d-b947-2cf7a93e4042', source='tool_agent', models_usage=RequestUsage(prompt_tokens=140, completion_tokens=61), metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 1, 22, 382090, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_F9po5NZambxVce4TlgAEsbeg', arguments='{"city": "서울"}', name='get_weather'), FunctionCall(id='call_ruqLuxw5pwFBqIhWJ6fwO7by', arguments='{"city": "부산"}', name='get_weather'), FunctionCall(id='call_3KPzmtuoTqH4BbapMysjipuj', arguments='{"expression": "234 * 567"}', name='calculate')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='c1868107-3fc7-4b39-a8ea-d76d1d0ab4a5', source='tool_agent', models_usage=None, metadata={},

### 7-2. `reflect_on_tool_use` 비교

`reflect_on_tool_use=False`일 때는 도구 결과가 그대로 반환되고, `True`일 때는 LLM이 자연어로 정리합니다.

In [8]:
# reflect_on_tool_use=False → 도구 결과 그대로 반환
raw_agent = AssistantAgent(
    name="raw_agent",
    model_client=model_client,
    tools=[get_weather],
    system_message="도구를 사용하여 날씨를 알려주세요.",
    reflect_on_tool_use=False,  # 도구 결과 그대로
)

print("=== reflect_on_tool_use=False ===")
result_raw = await Console(raw_agent.run_stream(task="제주 날씨 알려줘"))

print("\n=== reflect_on_tool_use=True ===")
# 위에서 만든 tool_agent는 reflect_on_tool_use=True
result_reflect = await Console(tool_agent.run_stream(task="제주 날씨 알려줘"))

=== reflect_on_tool_use=False ===
---------- TextMessage (user) ----------
제주 날씨 알려줘
---------- ToolCallRequestEvent (raw_agent) ----------
[FunctionCall(id='call_2IrarpM9Bchw3uWXtz2idHzj', arguments='{"city":"Jeju"}', name='get_weather')]
---------- ToolCallExecutionEvent (raw_agent) ----------
[FunctionExecutionResult(content='Jeju: 날씨 데이터 없음', name='get_weather', call_id='call_2IrarpM9Bchw3uWXtz2idHzj', is_error=False)]
---------- ToolCallSummaryMessage (raw_agent) ----------
Jeju: 날씨 데이터 없음

=== reflect_on_tool_use=True ===
---------- TextMessage (user) ----------
제주 날씨 알려줘
---------- ToolCallRequestEvent (tool_agent) ----------
[FunctionCall(id='call_juMBbwFaVUwk2VnkzuNUwJvx', arguments='{"city":"제주"}', name='get_weather')]
---------- ToolCallExecutionEvent (tool_agent) ----------
[FunctionExecutionResult(content='비, 17°C, 습도 85%', name='get_weather', call_id='call_juMBbwFaVUwk2VnkzuNUwJvx', is_error=False)]
---------- TextMessage (tool_agent) ----------
제주의 날씨는 비가 오고 있으며, 기온은 17°

### 7-3. Team에서 도구 전담 Agent 구성

여러 Agent가 각자 다른 도구를 담당하는 패턴입니다. SelectorGroupChat에서 LLM이 적합한 Agent를 선택합니다.

In [9]:
from autogen_agentchat.teams import SelectorGroupChat
from autogen_agentchat.conditions import TextMentionTermination, MaxMessageTermination


# 도구별 전담 Agent
weather_agent = AssistantAgent(
    "WeatherAgent",
    description="날씨 정보를 조회하는 에이전트. 날씨 관련 질문에 응답.",
    model_client=model_client,
    tools=[get_weather],
    system_message="날씨 도구를 사용하여 정확한 날씨 정보를 제공하세요.",
    reflect_on_tool_use=True,
)

calc_agent = AssistantAgent(
    "CalcAgent",
    description="수학 계산을 수행하는 에이전트. 계산 관련 질문에 응답.",
    model_client=model_client,
    tools=[calculate],
    system_message="계산 도구를 사용하여 정확한 결과를 제공하세요.",
    reflect_on_tool_use=True,
)

summarizer = AssistantAgent(
    "Summarizer",
    description="다른 에이전트의 결과를 종합하여 최종 답변을 작성하는 에이전트.",
    model_client=model_client,
    system_message="다른 에이전트의 결과를 종합하여 최종 답변을 작성하세요. 완료 시 'DONE'이라고 적으세요.",
)

termination = TextMentionTermination("DONE") | MaxMessageTermination(10)

team = SelectorGroupChat(
    [weather_agent, calc_agent, summarizer],
    model_client=model_client,
    termination_condition=termination,
)

await Console(team.run_stream(
    task="서울 날씨를 알려주고, 화씨로 변환해줘. (공식: °F = °C × 9/5 + 32)"
))

---------- TextMessage (user) ----------
서울 날씨를 알려주고, 화씨로 변환해줘. (공식: °F = °C × 9/5 + 32)
---------- ToolCallRequestEvent (WeatherAgent) ----------
[FunctionCall(id='call_HzgMrdJ9RTNHkzyFmhE0uxSz', arguments='{"city":"서울"}', name='get_weather')]
---------- ToolCallExecutionEvent (WeatherAgent) ----------
[FunctionExecutionResult(content='맑음, 22°C, 습도 45%', name='get_weather', call_id='call_HzgMrdJ9RTNHkzyFmhE0uxSz', is_error=False)]
---------- TextMessage (WeatherAgent) ----------
서울의 현재 날씨는 맑고 기온은 22°C입니다. 화씨로 변환하면 \( 22°C \times \frac{9}{5} + 32 = 71.6°F \)입니다.
---------- TextMessage (Summarizer) ----------
서울의 현재 날씨는 맑고 기온은 섭씨 22도입니다. 이를 화씨로 변환하면 71.6°F입니다. DONE


TaskResult(messages=[TextMessage(id='cadf11fd-a2fb-4c4a-a4c5-1c96124f0f04', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 1, 41, 439730, tzinfo=datetime.timezone.utc), content='서울 날씨를 알려주고, 화씨로 변환해줘. (공식: °F = °C × 9/5 + 32)', type='TextMessage'), ToolCallRequestEvent(id='6c85f6b7-61e0-457d-bd81-e183f831f5f6', source='WeatherAgent', models_usage=RequestUsage(prompt_tokens=104, completion_tokens=14), metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 1, 43, 904520, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_HzgMrdJ9RTNHkzyFmhE0uxSz', arguments='{"city":"서울"}', name='get_weather')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='a10d64ca-a52a-4522-8280-43831e40f519', source='WeatherAgent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 1, 43, 906278, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content='맑음, 22°C, 습도 45%', name='get_weather', call_id='call_Hzg

---
## 8. FunctionTool과 고급 도구 패턴

`FunctionTool` 클래스를 사용하면 도구의 이름과 설명을 커스터마이즈할 수 있습니다.
또한 도구 에러 처리, 도구 파이프라인 패턴을 살펴봅니다.

### 8-1. FunctionTool로 이름/설명 커스터마이즈

In [10]:
from autogen_core.tools import FunctionTool


# 원본 함수
async def translate(text: str, target_lang: str) -> str:
    """텍스트를 번역합니다."""
    # 실제로는 번역 API 호출
    translations = {
        "en": {"안녕하세요": "Hello", "감사합니다": "Thank you"},
        "ja": {"안녕하세요": "こんにちは", "감사합니다": "ありがとうございます"},
    }
    lang_data = translations.get(target_lang, {})
    result = lang_data.get(text, f"[{target_lang}] {text} (번역됨)")
    return result


# FunctionTool로 래핑 - 이름과 설명 커스터마이즈
translate_tool = FunctionTool(
    func=translate,
    name="translate_text",
    description="텍스트를 지정된 언어로 번역합니다. "
                "target_lang은 ISO 639-1 코드를 사용합니다: en(영어), ja(일본어), zh(중국어).",
)

# 스키마 확인
print(f"도구 이름: {translate_tool.name}")
print(f"도구 설명: {translate_tool.description}")
print(f"스키마: {translate_tool.schema}")

도구 이름: translate_text
도구 설명: 텍스트를 지정된 언어로 번역합니다. target_lang은 ISO 639-1 코드를 사용합니다: en(영어), ja(일본어), zh(중국어).
스키마: {'name': 'translate_text', 'description': '텍스트를 지정된 언어로 번역합니다. target_lang은 ISO 639-1 코드를 사용합니다: en(영어), ja(일본어), zh(중국어).', 'parameters': {'type': 'object', 'properties': {'text': {'description': 'text', 'title': 'Text', 'type': 'string'}, 'target_lang': {'description': 'target_lang', 'title': 'Target Lang', 'type': 'string'}}, 'required': ['text', 'target_lang'], 'additionalProperties': False}, 'strict': False}


In [11]:
# FunctionTool을 사용하는 Agent
translator_agent = AssistantAgent(
    name="translator",
    model_client=model_client,
    tools=[translate_tool],
    system_message="번역 도구를 사용하여 사용자의 번역 요청을 처리하세요.",
    reflect_on_tool_use=True,
)

await Console(translator_agent.run_stream(task="'안녕하세요'를 영어와 일본어로 번역해줘"))

---------- TextMessage (user) ----------
'안녕하세요'를 영어와 일본어로 번역해줘
---------- ToolCallRequestEvent (translator) ----------
[FunctionCall(id='call_jvlgR2scB45XXL7WtxuWxGIk', arguments='{"text": "안녕하세요", "target_lang": "en"}', name='translate_text'), FunctionCall(id='call_W29QHfGRZhnBArRUKqs0zug4', arguments='{"text": "안녕하세요", "target_lang": "ja"}', name='translate_text')]
---------- ToolCallExecutionEvent (translator) ----------
[FunctionExecutionResult(content='Hello', name='translate_text', call_id='call_jvlgR2scB45XXL7WtxuWxGIk', is_error=False), FunctionExecutionResult(content='こんにちは', name='translate_text', call_id='call_W29QHfGRZhnBArRUKqs0zug4', is_error=False)]
---------- TextMessage (translator) ----------
'안녕하세요'는 영어로 "Hello"이고, 일본어로 "こんにちは"입니다.


TaskResult(messages=[TextMessage(id='0aef6d59-9336-4868-bb21-aeee658b77ba', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 1, 54, 532793, tzinfo=datetime.timezone.utc), content="'안녕하세요'를 영어와 일본어로 번역해줘", type='TextMessage'), ToolCallRequestEvent(id='0e893196-6821-4a17-a34a-aa1334c6ce64', source='translator', models_usage=RequestUsage(prompt_tokens=129, completion_tokens=56), metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 1, 56, 323599, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_jvlgR2scB45XXL7WtxuWxGIk', arguments='{"text": "안녕하세요", "target_lang": "en"}', name='translate_text'), FunctionCall(id='call_W29QHfGRZhnBArRUKqs0zug4', arguments='{"text": "안녕하세요", "target_lang": "ja"}', name='translate_text')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='b0af573d-5b2b-4ff4-8cfe-05b4a80ac370', source='translator', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 1, 56, 326928, tzinf

### 8-2. 도구 에러 처리 패턴

도구에서 예외가 발생하면 **raise하지 말고 문자열로 반환**해야 LLM이 오류를 이해하고 대처할 수 있습니다.

In [12]:
# 에러 처리가 잘 된 도구 예시
async def fetch_stock_price(ticker: str) -> str:
    """주식 종목의 현재 가격을 조회합니다. ticker는 영문 심볼(예: AAPL, TSLA)입니다."""
    # 시뮬레이션 데이터
    stock_data = {
        "AAPL": {"price": 198.50, "change": 2.3},
        "TSLA": {"price": 245.20, "change": -1.5},
        "MSFT": {"price": 415.80, "change": 0.8},
    }

    ticker = ticker.upper().strip()
    if not ticker.isalpha():
        return f"오류: '{ticker}'는 유효한 티커 형식이 아닙니다. 영문 심볼을 사용해주세요."

    if ticker not in stock_data:
        return f"오류: '{ticker}' 종목을 찾을 수 없습니다. 사용 가능: {', '.join(stock_data.keys())}"

    data = stock_data[ticker]
    return f"{ticker}: ${data['price']:.2f} ({data['change']:+.2f}%)"


# 에러 처리 테스트
error_agent = AssistantAgent(
    name="stock_agent",
    model_client=model_client,
    tools=[fetch_stock_price],
    system_message="주식 가격 조회 도구를 사용하세요. 오류가 발생하면 사용자에게 안내하세요.",
    reflect_on_tool_use=True,
)

# 존재하지 않는 티커 → 에러 메시지를 LLM이 이해하고 대처
await Console(error_agent.run_stream(task="GOOGL 주가를 알려줘"))

---------- TextMessage (user) ----------
GOOGL 주가를 알려줘
---------- ToolCallRequestEvent (stock_agent) ----------
[FunctionCall(id='call_gqOgFW9unJu5UC4vbt8IFs7C', arguments='{"ticker":"GOOGL"}', name='fetch_stock_price')]
---------- ToolCallExecutionEvent (stock_agent) ----------
[FunctionExecutionResult(content="오류: 'GOOGL' 종목을 찾을 수 없습니다. 사용 가능: AAPL, TSLA, MSFT", name='fetch_stock_price', call_id='call_gqOgFW9unJu5UC4vbt8IFs7C', is_error=False)]
---------- TextMessage (stock_agent) ----------
죄송합니다. 현재 Google 주식(GOOGL)의 주가 정보를 제공할 수 없습니다. 대신 Apple(AAPL), Tesla(TSLA), Microsoft(MSFT)의 주가 정보를 제공해 드릴 수 있습니다. 다른 정보를 원하시면 말씀해 주세요.


TaskResult(messages=[TextMessage(id='4a44a288-d9d2-4ea6-849b-93792e861b42', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 2, 4, 216140, tzinfo=datetime.timezone.utc), content='GOOGL 주가를 알려줘', type='TextMessage'), ToolCallRequestEvent(id='97672b17-2b2e-4563-9e25-8ec627cd6745', source='stock_agent', models_usage=RequestUsage(prompt_tokens=101, completion_tokens=17), metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 2, 5, 830, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_gqOgFW9unJu5UC4vbt8IFs7C', arguments='{"ticker":"GOOGL"}', name='fetch_stock_price')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='79492f1b-cec5-485f-9ebf-5c8066a763fa', source='stock_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 2, 5, 6999, tzinfo=datetime.timezone.utc), content=[FunctionExecutionResult(content="오류: 'GOOGL' 종목을 찾을 수 없습니다. 사용 가능: AAPL, TSLA, MSFT", name='fetch_stock_price', call_id='c

### 8-3. 도구 파이프라인: Agent가 여러 도구를 순차 호출

하나의 Agent에 여러 도구를 등록하면, LLM이 자율적으로 순서를 정해 호출합니다.

In [13]:
# 여러 도구를 가진 Agent → 자율적으로 파이프라인 구성
pipeline_agent = AssistantAgent(
    name="pipeline_agent",
    model_client=model_client,
    tools=[fetch_stock_price, calculate, translate_tool],
    system_message="도구를 조합하여 복잡한 요청을 처리하세요. 한국어로 답변하세요.",
    reflect_on_tool_use=True,
)

# Agent가 스스로 도구 호출 순서를 결정:
# 1) fetch_stock_price("AAPL") → 가격 확인
# 2) fetch_stock_price("TSLA") → 가격 확인
# 3) calculate("198.50 + 245.20") → 합계 계산
await Console(pipeline_agent.run_stream(
    task="AAPL과 TSLA 주가를 조회하고, 두 종목의 가격 합계를 계산해줘"
))

---------- TextMessage (user) ----------
AAPL과 TSLA 주가를 조회하고, 두 종목의 가격 합계를 계산해줘
---------- ToolCallRequestEvent (pipeline_agent) ----------
[FunctionCall(id='call_h7WJ315RWpxDNBPX7BbLIT79', arguments='{"ticker": "AAPL"}', name='fetch_stock_price'), FunctionCall(id='call_2PpQdVmaVqxVS0B6de8BTs9u', arguments='{"ticker": "TSLA"}', name='fetch_stock_price')]
---------- ToolCallExecutionEvent (pipeline_agent) ----------
[FunctionExecutionResult(content='AAPL: $198.50 (+2.30%)', name='fetch_stock_price', call_id='call_h7WJ315RWpxDNBPX7BbLIT79', is_error=False), FunctionExecutionResult(content='TSLA: $245.20 (-1.50%)', name='fetch_stock_price', call_id='call_2PpQdVmaVqxVS0B6de8BTs9u', is_error=False)]
---------- TextMessage (pipeline_agent) ----------
현재 AAPL 주가는 $198.50이며, TSLA 주가는 $245.20입니다. 두 종목의 주가를 합산하면 $443.70입니다.


TaskResult(messages=[TextMessage(id='8ba8073c-08a5-474e-b7c8-c917dfbd390b', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 2, 48, 491459, tzinfo=datetime.timezone.utc), content='AAPL과 TSLA 주가를 조회하고, 두 종목의 가격 합계를 계산해줘', type='TextMessage'), ToolCallRequestEvent(id='4d51a681-3653-4083-b490-1831d290eab0', source='pipeline_agent', models_usage=RequestUsage(prompt_tokens=237, completion_tokens=48), metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 2, 50, 373721, tzinfo=datetime.timezone.utc), content=[FunctionCall(id='call_h7WJ315RWpxDNBPX7BbLIT79', arguments='{"ticker": "AAPL"}', name='fetch_stock_price'), FunctionCall(id='call_2PpQdVmaVqxVS0B6de8BTs9u', arguments='{"ticker": "TSLA"}', name='fetch_stock_price')], type='ToolCallRequestEvent'), ToolCallExecutionEvent(id='95bab1ba-6b31-4ca9-bc98-a5bc5a37f83f', source='pipeline_agent', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 2, 50, 378030, tzinfo=datetime

---
## 9. Human-in-the-Loop (사람 개입)

에이전트 대화에 사람이 개입하는 두 가지 전략:
- **전략 A**: `UserProxyAgent`를 Team에 포함하여 대화 중 개입
- **전략 B**: `max_turns=1`로 실행 후 루프에서 피드백

### 9-1. 전략 B: 실행 후 피드백 루프

> 노트북 환경에서는 `input()` 대신 고정 피드백을 사용합니다.

In [14]:
from autogen_agentchat.teams import RoundRobinGroupChat

poet = AssistantAgent(
    "poet",
    model_client=model_client,
    system_message="당신은 시인입니다. 주어진 주제로 4줄 시를 작성하세요. 피드백을 반영하여 수정하세요.",
)

# max_turns=1: Agent가 1번만 발언하고 멈춤
team = RoundRobinGroupChat([poet], max_turns=1)

# 시뮬레이션된 피드백 루프 (실제로는 input() 사용)
feedbacks = [
    "바다를 주제로 시를 써줘",
    "운율을 맞춰서 다시 써줘",
    "마지막 줄을 더 감동적으로 바꿔줘",
]

for i, feedback in enumerate(feedbacks):
    print(f"\n{'='*50}")
    print(f"[사용자 입력 {i+1}] {feedback}")
    print('='*50)
    result = await Console(team.run_stream(task=feedback))


[사용자 입력 1] 바다를 주제로 시를 써줘
---------- TextMessage (user) ----------
바다를 주제로 시를 써줘
---------- TextMessage (poet) ----------
끝없는 파도 속에 꿈을 싣고,  
저 멀리 수평선은 희망의 벽,  
은빛 물결 위에 추억을 새기며,  
바다는 조용히 우리를 위로하네.  

[사용자 입력 2] 운율을 맞춰서 다시 써줘
---------- TextMessage (user) ----------
운율을 맞춰서 다시 써줘
---------- TextMessage (poet) ----------
끝없는 파도에 꿈을 담아,  
수평선 너머로 희망 찾아.  
은빛 물결 위에 추억을 새겨,  
바다는 조용히 마음을 어루만져.  

[사용자 입력 3] 마지막 줄을 더 감동적으로 바꿔줘
---------- TextMessage (user) ----------
마지막 줄을 더 감동적으로 바꿔줘
---------- TextMessage (poet) ----------
끝없는 파도에 꿈을 담아,  
수평선 너머로 희망 찾아.  
은빛 물결 위에 추억을 새겨,  
바다는 고요히 영혼을 감싸네.  


### 9-2. 전략 A: UserProxyAgent로 대화 중 개입

`UserProxyAgent`를 Team에 포함하면 Agent 차례마다 사람 입력을 기다립니다.

> 노트북에서는 커스텀 `input_func`으로 시뮬레이션합니다.

In [15]:
from autogen_agentchat.agents import UserProxyAgent

# 시뮬레이션된 사용자 입력 (실제로는 input 사용)
simulated_inputs = iter([
    "좀 더 짧게 3줄로 줄여줘",
    "APPROVE",  # 종료 트리거
])


async def simulated_input(prompt: str, cancellation_token=None) -> str:
    """시뮬레이션된 사용자 입력"""
    response = next(simulated_inputs)
    print(f"  [시뮬레이션 입력] {response}")
    return response


assistant = AssistantAgent(
    "assistant",
    model_client=model_client,
    system_message="사용자의 요청에 따라 글을 작성하고 수정하세요.",
)

user_proxy = UserProxyAgent("user_proxy", input_func=simulated_input)

termination = TextMentionTermination("APPROVE")
team = RoundRobinGroupChat(
    [assistant, user_proxy],
    termination_condition=termination,
)

# assistant → user_proxy(입력 대기) → assistant → user_proxy → ...
await Console(team.run_stream(task="봄을 주제로 5줄 시를 써줘"))

---------- TextMessage (user) ----------
봄을 주제로 5줄 시를 써줘
---------- TextMessage (assistant) ----------
  [시뮬레이션 입력] 좀 더 짧게 3줄로 줄여줘
봄의 속삭임, 부드러운 바람,  
장미의 향기, 따사로운 햇살,  
초록의 세상, 피어나는 새싹,  
희망의 계절, 마음을 깨우네,  
벌들의 춤, 사랑을 전하는 봄.  
---------- TextMessage (user_proxy) ----------
좀 더 짧게 3줄로 줄여줘
---------- TextMessage (assistant) ----------
  [시뮬레이션 입력] APPROVE
봄바람 속삭임, 새싹의 춤,  
장미 향 가득한 희망의 계절,  
햇살 아래 사랑 싹트네.  
---------- TextMessage (user_proxy) ----------
APPROVE


TaskResult(messages=[TextMessage(id='56a8138d-850b-4e4b-9add-bc8843a06ee9', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 3, 4, 749871, tzinfo=datetime.timezone.utc), content='봄을 주제로 5줄 시를 써줘', type='TextMessage'), TextMessage(id='2ab3ff85-323a-49df-bded-d6043af99cb3', source='assistant', models_usage=RequestUsage(prompt_tokens=37, completion_tokens=68), metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 3, 6, 560687, tzinfo=datetime.timezone.utc), content='봄의 속삭임, 부드러운 바람,  \n장미의 향기, 따사로운 햇살,  \n초록의 세상, 피어나는 새싹,  \n희망의 계절, 마음을 깨우네,  \n벌들의 춤, 사랑을 전하는 봄.  ', type='TextMessage'), UserInputRequestedEvent(id='92d0277e-b067-482b-a3c3-4a926d8e8986', source='user_proxy', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 3, 6, 563729, tzinfo=datetime.timezone.utc), request_id='bbb06edc-289a-4bb3-91ba-ccf40d04b58d', content='', type='UserInputRequestedEvent'), TextMessage(id='02c5aa37-8f3e-4ff3-bb95-afa1a6938cc6', s

### 9-3. Swarm에서의 사용자 핸드오프

Swarm 패턴에서 중요한 결정 전에 사용자 승인을 받는 패턴입니다.

In [16]:
from autogen_agentchat.teams import Swarm
from autogen_agentchat.conditions import HandoffTermination
from autogen_agentchat.messages import HandoffMessage


# 도구: 환불 처리
async def refund_flight(flight_id: str) -> str:
    """항공편을 환불 처리합니다."""
    return f"항공편 {flight_id} 환불이 완료되었습니다. 환불 금액: 350,000원"


travel_agent = AssistantAgent(
    "travel_agent",
    model_client=model_client,
    handoffs=["flights_refunder", "user"],
    system_message="""당신은 여행 상담 에이전트입니다.
    - 환불 요청은 flights_refunder에게 위임하세요.
    - 사용자 정보가 필요하면 user에게 핸드오프하세요.
    - 완료 시 TERMINATE라고 응답하세요.""",
)

flights_refunder = AssistantAgent(
    "flights_refunder",
    model_client=model_client,
    handoffs=["travel_agent"],
    tools=[refund_flight],
    system_message="""환불 전문 에이전트입니다. refund_flight 도구로 환불을 처리하세요.
    완료 후 travel_agent에게 핸드오프하세요.""",
)

termination = (
    HandoffTermination(target="user")
    | TextMentionTermination("TERMINATE")
    | MaxMessageTermination(15)
)

swarm_team = Swarm(
    [travel_agent, flights_refunder],
    termination_condition=termination,
)

# 1단계: 초기 요청 → user에게 핸드오프 (정보 요청)
print("=== 1단계: 초기 요청 ===")
result = await Console(swarm_team.run_stream(task="항공편 환불을 요청합니다"))

# 2단계: 사용자가 정보를 제공하며 재개
last_msg = result.messages[-1]
if isinstance(last_msg, HandoffMessage) and last_msg.target == "user":
    print("\n=== 2단계: 사용자 정보 제공 후 재개 ===")
    result = await Console(swarm_team.run_stream(
        task=HandoffMessage(
            source="user",
            target=last_msg.source,
            content="항공편 ID는 FL-12345입니다."
        )
    ))

=== 1단계: 초기 요청 ===
---------- TextMessage (user) ----------
항공편 환불을 요청합니다
---------- ToolCallRequestEvent (travel_agent) ----------
[FunctionCall(id='call_rzmsxIG6XB26skHEqGM7W3gf', arguments='{}', name='transfer_to_flights_refunder')]
---------- ToolCallExecutionEvent (travel_agent) ----------
[FunctionExecutionResult(content='Transferred to flights_refunder, adopting the role of flights_refunder immediately.', name='transfer_to_flights_refunder', call_id='call_rzmsxIG6XB26skHEqGM7W3gf', is_error=False)]
---------- HandoffMessage (travel_agent) ----------
Transferred to flights_refunder, adopting the role of flights_refunder immediately.
---------- TextMessage (flights_refunder) ----------
환불 처리에 필요한 항공편 ID를 알려주시면 도와드리겠습니다.
---------- ToolCallRequestEvent (flights_refunder) ----------
[FunctionCall(id='call_pGZfxnXxnYdYegmQe9LDkKlq', arguments='{}', name='transfer_to_travel_agent')]
---------- ToolCallExecutionEvent (flights_refunder) ----------
[FunctionExecutionResult(content='Trans

---
## 10. Termination 조건

Team 대화의 종료 시점을 결정하는 다양한 조건들을 조합하여 사용할 수 있습니다.

### 10-1. 다양한 Termination 조건 테스트

In [17]:
from autogen_agentchat.conditions import (
    MaxMessageTermination,
    TextMentionTermination,
    TimeoutTermination,
)

# === 테스트 1: MaxMessageTermination ===
print("=== MaxMessageTermination(5) 테스트 ===")

agent_a = AssistantAgent(
    "agent_a", model_client=model_client,
    system_message="간단하게 한 문장으로만 답하세요.",
)
agent_b = AssistantAgent(
    "agent_b", model_client=model_client,
    system_message="상대방의 말에 한 문장으로 반박하세요.",
)

max_term = MaxMessageTermination(5)  # 5개 메시지 후 종료
team1 = RoundRobinGroupChat([agent_a, agent_b], termination_condition=max_term)
result = await Console(team1.run_stream(task="AI가 인간을 대체할 수 있을까?"))
print(f"\n종료 사유: {result.stop_reason}")
print(f"총 메시지 수: {len(result.messages)}")

=== MaxMessageTermination(5) 테스트 ===
---------- TextMessage (user) ----------
AI가 인간을 대체할 수 있을까?
---------- TextMessage (agent_a) ----------
일부 작업에서 가능하지만, 인간의 고유한 창의성, 감성, 도덕적 판단은 대체하기 어려울 것입니다.
---------- TextMessage (agent_b) ----------
AI는 특정 작업에서 인간의 능력을 대체할 수 있어도, 복잡한 감정적 및 윤리적 판단은 인간 고유의 영역입니다.
---------- TextMessage (agent_a) ----------
맞습니다, AI는 특정 영역에서 인간을 보조하거나 대체할 수 있지만, 인간의 감정적, 윤리적 판단은 대체하기 어렵습니다.
---------- TextMessage (agent_b) ----------
AI는 특정 영역에서는 효과적이지만, 인간의 감정이나 윤리적 판단 능력을 완전히 대체할 수 없습니다.

종료 사유: Maximum number of messages 5 reached, current message count: 5
총 메시지 수: 5


In [18]:
# === 테스트 2: TextMentionTermination ===
print("=== TextMentionTermination('APPROVE') 테스트 ===")

writer = AssistantAgent(
    "writer", model_client=model_client,
    system_message="주어진 주제로 짧은 글을 작성하세요.",
)
critic = AssistantAgent(
    "critic", model_client=model_client,
    system_message="글을 검토하세요. 만족스러우면 'APPROVE'라고만 응답하세요.",
)

text_term = TextMentionTermination("APPROVE") | MaxMessageTermination(10)
team2 = RoundRobinGroupChat([writer, critic], termination_condition=text_term)
result = await Console(team2.run_stream(task="커피의 장점 3가지"))
print(f"\n종료 사유: {result.stop_reason}")

=== TextMentionTermination('APPROVE') 테스트 ===
---------- TextMessage (user) ----------
커피의 장점 3가지
---------- TextMessage (writer) ----------
커피는 전 세계적으로 사랑받는 음료로, 그 매력은 다양한 장점을 통해 나타납니다. 첫째, 커피는 에너지를 제공하는 강력한 도구입니다. 카페인 함량 덕분에 신경을 자극하여 피로를 감소시키고 집중력을 향상시킵니다. 아침에 눈을 뜨기 어렵거나 오후의 나른함을 극복해야 할 때 한 잔의 커피가 큰 도움이 될 수 있습니다.

둘째, 커피는 뛰어난 항산화제를 포함하고 있습니다. 많은 연구들이 커피에 포함된 항산화 물질이 체내 활성산소를 제거하여 세포 손상을 방지하고 노화를 억제하는 데 도움이 된다고 보고하고 있습니다. 이러한 항산화 효과는 심혈관 질환 발병 위험을 줄이고, 장기적으로 건강한 삶을 유지하는 데 기여할 수 있습니다.

마지막으로, 커피는 사회적 연결을 촉진하는 역할을 합니다. 커피숍에서 친구들과 만남을 가지거나, 직장에서 커피 브레이크를 통해 동료와 소통하는 것은 중요한 사회적 상호작용의 기회를 제공합니다. 또한, 전 세계적으로 만연한 커피 문화는 국가 간의 교류를 촉진하기도 합니다.

이처럼 커피는 단순한 음료를 넘어 신체적, 정신적, 사회적 측면에서 다양한 장점을 지니고 있습니다.
---------- TextMessage (critic) ----------
APPROVE

종료 사유: Text 'APPROVE' mentioned


### 10-2. OR / AND 조합

- `|` (OR): 하나라도 충족되면 종료
- `&` (AND): 모두 충족되어야 종료

In [19]:
# === OR 조합: 안전 장치 패턴 ===
# 정상 종료(APPROVE) 또는 메시지 상한(20) 또는 시간 초과(60초)
safe_termination = (
    TextMentionTermination("APPROVE")
    | MaxMessageTermination(20)
    | TimeoutTermination(timeout_seconds=60)
)
print(f"OR 조합 생성 완료: {safe_termination}")

# === AND 조합: 엄격한 종료 ===
# 메시지 10개 이상 AND 'DONE' 키워드 언급 → 둘 다 충족해야 종료
strict_termination = (
    MaxMessageTermination(10)
    & TextMentionTermination("DONE")
)
print(f"AND 조합 생성 완료: {strict_termination}")

# AND + OR 혼합
# (APPROVE가 나오면 즉시 종료) 또는 (메시지 10개 이상 AND DONE)
mixed_termination = (
    TextMentionTermination("APPROVE")
    | (MaxMessageTermination(10) & TextMentionTermination("DONE"))
)
print(f"혼합 조합 생성 완료: {mixed_termination}")

OR 조합 생성 완료: <autogen_agentchat.base._termination.OrTerminationCondition object at 0x12a3b5910>
AND 조합 생성 완료: <autogen_agentchat.base._termination.AndTerminationCondition object at 0x12a3b02f0>
혼합 조합 생성 완료: <autogen_agentchat.base._termination.OrTerminationCondition object at 0x12a37b790>


### 10-3. Team 상태 관리: reset과 대화 재개

In [20]:
# Team 상태 관리 데모
simple_agent = AssistantAgent(
    "simple", model_client=model_client,
    system_message="간결하게 한국어로 답하세요.",
)

team3 = RoundRobinGroupChat(
    [simple_agent],
    max_turns=1,
)

# 1차 실행
print("=== 1차 실행 ===")
result1 = await Console(team3.run_stream(task="내 이름은 민수야. 기억해줘."))

# 2차 실행 (이전 대화 이어감 - reset 안 함)
print("\n=== 2차 실행 (대화 이어감) ===")
result2 = await Console(team3.run_stream(task="내 이름이 뭐라고 했지?"))

# reset 후 3차 실행 (새로운 대화)
await team3.reset()
print("\n=== 3차 실행 (reset 후 새 대화) ===")
result3 = await Console(team3.run_stream(task="내 이름이 뭐라고 했지?"))

=== 1차 실행 ===
---------- TextMessage (user) ----------
내 이름은 민수야. 기억해줘.
---------- TextMessage (simple) ----------
죄송하지만, 개인 정보를 기억할 수 없습니다. 어떻게 도와드릴까요?

=== 2차 실행 (대화 이어감) ===
---------- TextMessage (user) ----------
내 이름이 뭐라고 했지?
---------- TextMessage (simple) ----------
죄송하지만, 저는 개인 정보를 기억할 수 없습니다.

=== 3차 실행 (reset 후 새 대화) ===
---------- TextMessage (user) ----------
내 이름이 뭐라고 했지?
---------- TextMessage (simple) ----------
죄송하지만, 당신의 이름을 알 수 없습니다.


---
## 11. 에이전트 상태 저장과 복원

`save_state()` / `load_state()`로 Team과 Agent의 상태를 JSON으로 저장하고, 새 세션에서 복원할 수 있습니다.

### 11-1. Team 상태 저장/복원

In [21]:
import json

# 1. Team 생성 및 첫 실행
writer_s = AssistantAgent(
    "writer", model_client=model_client,
    system_message="주어진 주제로 짧은 에세이를 작성하세요.",
)
critic_s = AssistantAgent(
    "critic", model_client=model_client,
    system_message="에세이를 검토하고 피드백을 주세요. 만족스러우면 'APPROVE'라고 응답하세요.",
)

term = TextMentionTermination("APPROVE") | MaxMessageTermination(6)
team_save = RoundRobinGroupChat([writer_s, critic_s], termination_condition=term)

print("=== 1단계: 첫 실행 ===")
result = await Console(team_save.run_stream(task="AI 윤리에 대한 짧은 에세이를 써줘"))

# 2. 상태 저장
state = await team_save.save_state()
with open("team_state.json", "w") as f:
    json.dump(state, f, ensure_ascii=False, indent=2)
print(f"\n상태 저장 완료! (파일 크기: {len(json.dumps(state))} bytes)")

=== 1단계: 첫 실행 ===
---------- TextMessage (user) ----------
AI 윤리에 대한 짧은 에세이를 써줘
---------- TextMessage (writer) ----------
AI 윤리는 인공지능 기술의 발전과 사용에 관한 도덕적 문제와 책임을 다루는 분야로, 현대 사회의 급변하는 기술 환경 속에서 점점 더 중요해지고 있다. 인공지능은 의료, 금융, 자율주행차 등 여러 분야에서 혁신을 가져오고 있으나, 그에 따른 윤리적 문제가 대두되고 있다. 가장 흔한 걱정 중 하나는 개인정보 보호와 관련된 문제로, AI가 방대한 데이터를 처리하며 개인의 사생활을 침해할 가능성이 있다는 점이다. 따라서, 개인정보 보호를 위한 강력한 규제와 감시가 반드시 필요하다.

또 다른 중요한 이슈는 AI 의사 결정의 투명성과 공정성이다. AI 시스템의 알고리즘이 편향된 데이터를 기반으로 학습할 경우, 결과 역시 편향될 가능성이 크다. 이는 특정 집단에 대한 차별이나 불합리한 결과를 초래할 수 있다. 따라서 AI 시스템 개발자는 알고리즘의 공정성을 검토하고, 다양한 관점을 반영하는 데이터를 사용하여 훈련시켜야 한다.

더불어, AI 기술로 인한 일자리 감소 문제도 윤리적 논의의 중요한 부분이다. 자동화로 인해 전통적인 직업들이 사라질 수 있으며, 이는 경제적 불평등을 심화시키는 결과로 이어질 수 있다. 사회는 이러한 변화에 대비하여 새로운 교육 및 직업 기회를 마련하고, 변화에 적응할 수 있는 시스템을 구축해야 할 것이다.

AI 윤리는 인류 사회에 인공지능이 미칠 영향을 고려하며 지속 가능한 발전 방안을 모색하는 데 그 목적이 있다. 기술의 발전이 모두에게 이익이 되도록 하기 위해서는 정부, 기업, 연구 기관 등이 협력하여 올바른 윤리적 기준을 정하고 준수할 필요가 있다. AI 윤리는 단순히 현재의 문제를 해결하기 위한 것이 아니라, 인공지능이 긍정적인 방향으로 발전하는 데 필수적인 밑거름이 되어야 한다. 인류의 이익을 최우선으로 삼는 윤리적

In [22]:
# 3. 새 Team 인스턴스에 상태 복원 (새 세션 시뮬레이션)
writer_s2 = AssistantAgent(
    "writer", model_client=model_client,
    system_message="주어진 주제로 짧은 에세이를 작성하세요.",
)
critic_s2 = AssistantAgent(
    "critic", model_client=model_client,
    system_message="에세이를 검토하고 피드백을 주세요. 만족스러우면 'APPROVE'라고 응답하세요.",
)

term2 = TextMentionTermination("APPROVE") | MaxMessageTermination(6)
team_restored = RoundRobinGroupChat([writer_s2, critic_s2], termination_condition=term2)

# 상태 파일에서 복원
with open("team_state.json", "r") as f:
    saved_state = json.load(f)
await team_restored.load_state(saved_state)

# 4. 이전 대화를 이어서 진행
print("=== 2단계: 상태 복원 후 대화 재개 ===")
result = await Console(team_restored.run_stream(task="결론 부분을 좀 더 강하게 수정해줘"))
print(f"\n종료 사유: {result.stop_reason}")

=== 2단계: 상태 복원 후 대화 재개 ===
---------- TextMessage (user) ----------
결론 부분을 좀 더 강하게 수정해줘
---------- TextMessage (writer) ----------
AI 윤리는 인류 사회에 인공지능이 미칠 영향을 깊이 있게 고려하며, 지속 가능한 발전 방안을 모색하는 데 그 목적을 두고 있다. 기술의 발전이 단지 소수에게만 혜택을 주는 것이 아니라 모든 사람에게 이익이 되도록 하기 위해서는, 정부와 기업, 연구 기관, 그리고 일반 대중까지 모두가 협력하여 올바른 윤리적 기준을 정하고 이를 엄격히 준수해야 한다. 더 이상 윤리를 선택 사항으로 여길 수 없다. AI 윤리는 인류의 존엄과 평등을 수호하기 위한 필수 조건이자, 우리의 미래를 결정짓는 중대 요소로 자리 잡아야 한다. 우리는 지금 이 순간부터 미래를 준비하며 기술이 아닌 인간 중심의 관점에서 AI를 평가하고 다루어야 한다. 진정한 발전은 기술의 힘이 인류의 공익을 증진시키는 데 사용될 때 비로소 이루어질 수 있다. 이 강력한 윤리적 기반 위에서 우리는 AI의 잠재력을 최대한 긍정적으로 활용하고, 지속 가능한 미래를 구축할 수 있을 것이다.
---------- TextMessage (critic) ----------
APPROVE

종료 사유: Text 'APPROVE' mentioned


### 11-2. 개별 Agent 상태 저장/복원

In [23]:
# 개별 Agent 상태 저장
agent_orig = AssistantAgent(
    "memory_agent", model_client=model_client,
    system_message="간결하게 답하세요.",
)

# 대화 기록 생성
team_a = RoundRobinGroupChat([agent_orig], max_turns=1)
await Console(team_a.run_stream(task="내 이름은 영희, 좋아하는 색은 파란색이야."))
await Console(team_a.run_stream(task="내 취미는 독서야."))

# Agent 상태 저장
agent_state = await agent_orig.save_state()
print(f"Agent 상태 키: {list(agent_state.keys())}")

# 새 Agent 인스턴스에 복원
agent_new = AssistantAgent(
    "memory_agent", model_client=model_client,
    system_message="간결하게 답하세요.",
)
await agent_new.load_state(agent_state)

# 복원된 Agent로 확인
team_b = RoundRobinGroupChat([agent_new], max_turns=1)
print("\n=== 복원된 Agent에게 질문 ===")
await Console(team_b.run_stream(task="내 이름, 좋아하는 색, 취미가 뭐라고 했지?"))

---------- TextMessage (user) ----------
내 이름은 영희, 좋아하는 색은 파란색이야.
---------- TextMessage (memory_agent) ----------
반가워요, 영희 씨! 파란색은 참 멋진 색이죠.
---------- TextMessage (user) ----------
내 취미는 독서야.
---------- TextMessage (memory_agent) ----------
멋진 취미네요! 어떤 종류의 책을 주로 읽으세요?
Agent 상태 키: ['type', 'version', 'llm_context']

=== 복원된 Agent에게 질문 ===
---------- TextMessage (user) ----------
내 이름, 좋아하는 색, 취미가 뭐라고 했지?
---------- TextMessage (memory_agent) ----------
당신의 이름은 영희, 좋아하는 색은 파란색, 취미는 독서라고 하셨어요.


TaskResult(messages=[TextMessage(id='422cb0c1-2acc-4a41-9ffc-991283d66211', source='user', models_usage=None, metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 5, 17, 442979, tzinfo=datetime.timezone.utc), content='내 이름, 좋아하는 색, 취미가 뭐라고 했지?', type='TextMessage'), TextMessage(id='c126246b-df74-4440-b988-e0f92afc0906', source='memory_agent', models_usage=RequestUsage(prompt_tokens=114, completion_tokens=27), metadata={}, created_at=datetime.datetime(2026, 4, 7, 11, 5, 18, 126862, tzinfo=datetime.timezone.utc), content='당신의 이름은 영희, 좋아하는 색은 파란색, 취미는 독서라고 하셨어요.', type='TextMessage')], stop_reason='Maximum number of turns 1 reached.')

---
## 12. 디버깅과 모니터링

`TaskResult` 객체를 분석하여 대화 흐름, 종료 사유, 토큰 사용량을 추적합니다.

### 12-1. 실행 결과(TaskResult) 분석

In [24]:
# 분석용 Team 실행
planner_d = AssistantAgent(
    "Planner",
    description="작업을 계획하는 에이전트.",
    model_client=model_client,
    system_message="작업을 3단계로 분해하세요.",
)
executor_d = AssistantAgent(
    "Executor",
    description="계획을 실행하는 에이전트.",
    model_client=model_client,
    system_message="계획을 실행하고 결과를 보고하세요. 완료 시 'DONE'이라고 적으세요.",
)

term_d = TextMentionTermination("DONE") | MaxMessageTermination(6)
team_d = SelectorGroupChat(
    [planner_d, executor_d],
    model_client=model_client,
    termination_condition=term_d,
)

result = await team_d.run(task="Python으로 피보나치 함수를 작성하는 계획을 세워줘")

# === 결과 분석 ===
print("=" * 60)
print("대화 흐름 분석")
print("=" * 60)

for i, msg in enumerate(result.messages):
    content_preview = msg.content[:80] if isinstance(msg.content, str) else str(msg.content)[:80]
    print(f"  [{i}] {msg.source:12s} | {type(msg).__name__:20s} | {content_preview}...")

print(f"\n총 메시지 수: {len(result.messages)}")
print(f"종료 사유: {result.stop_reason}")

대화 흐름 분석
  [0] user         | TextMessage          | Python으로 피보나치 함수를 작성하는 계획을 세워줘...
  [1] Planner      | TextMessage          | 피보나치 함수를 작성하기 위한 계획은 다음의 3단계로 분해할 수 있습니다.

### 단계 1: 기본 이해 및 계획 수립
- **피보나치 수열 이...
  [2] Executor     | TextMessage          | 계획에 따라 피보나치 함수를 작성하고 테스트를 수행하겠습니다. 

```python
# 재귀적 접근법을 통한 피보나치 함수
def fib_rec...

총 메시지 수: 3
종료 사유: Text 'DONE' mentioned


### 12-2. 토큰 사용량 추적

In [25]:
# 토큰 사용량 분석
print("=" * 60)
print("토큰 사용량 분석")
print("=" * 60)

total_prompt = 0
total_completion = 0

for msg in result.messages:
    if hasattr(msg, "models_usage") and msg.models_usage:
        usage = msg.models_usage
        prompt_tokens = usage.prompt_tokens
        completion_tokens = usage.completion_tokens
        total_prompt += prompt_tokens
        total_completion += completion_tokens
        print(f"  [{msg.source:12s}] 입력: {prompt_tokens:,}  출력: {completion_tokens:,}")

print(f"\n총 입력 토큰: {total_prompt:,}")
print(f"총 출력 토큰: {total_completion:,}")
print(f"총 토큰: {total_prompt + total_completion:,}")

# GPT-4o 기준 비용 추정 (2024년 기준)
cost = (total_prompt * 2.5 / 1_000_000) + (total_completion * 10 / 1_000_000)
print(f"예상 비용 (GPT-4o): ${cost:.4f}")

토큰 사용량 분석
  [Planner     ] 입력: 39  출력: 516
  [Executor    ] 입력: 569  출력: 441

총 입력 토큰: 608
총 출력 토큰: 957
총 토큰: 1,565
예상 비용 (GPT-4o): $0.0111


### 12-3. 로깅 설정

AutoGen 내부 로그를 활성화하여 Agent 선택, 도구 호출 등의 상세 과정을 확인합니다.

In [26]:
import logging

# AutoGen 로깅 활성화
logging.basicConfig(level=logging.WARNING)
logging.getLogger("autogen_agentchat").setLevel(logging.DEBUG)

# 로그가 활성화된 상태에서 실행 → 내부 동작을 상세히 확인
debug_agent = AssistantAgent(
    "debug_agent", model_client=model_client,
    tools=[get_weather],
    system_message="도구를 사용하세요.",
    reflect_on_tool_use=True,
)

debug_team = RoundRobinGroupChat([debug_agent], max_turns=1)
await Console(debug_team.run_stream(task="서울 날씨 알려줘"))

# 로깅 원복
logging.getLogger("autogen_agentchat").setLevel(logging.WARNING)

---------- TextMessage (user) ----------
서울 날씨 알려줘


DEBUG:autogen_agentchat.events:id='a6d16435-b0c3-4f52-bcc0-82048f4d494d' source='debug_agent' models_usage=RequestUsage(prompt_tokens=66, completion_tokens=14) metadata={} created_at=datetime.datetime(2026, 4, 7, 11, 5, 32, 104013, tzinfo=datetime.timezone.utc) content=[FunctionCall(id='call_2dXBsmW0gw7enfR37799JWtk', arguments='{"city":"서울"}', name='get_weather')] type='ToolCallRequestEvent'
DEBUG:autogen_agentchat.events:id='9dca98c5-4190-4840-b7e0-6e6e84c9e8f4' source='debug_agent' models_usage=None metadata={} created_at=datetime.datetime(2026, 4, 7, 11, 5, 32, 105987, tzinfo=datetime.timezone.utc) content=[FunctionExecutionResult(content='맑음, 22°C, 습도 45%', name='get_weather', call_id='call_2dXBsmW0gw7enfR37799JWtk', is_error=False)] type='ToolCallExecutionEvent'


---------- ToolCallRequestEvent (debug_agent) ----------
[FunctionCall(id='call_2dXBsmW0gw7enfR37799JWtk', arguments='{"city":"서울"}', name='get_weather')]
---------- ToolCallExecutionEvent (debug_agent) ----------
[FunctionExecutionResult(content='맑음, 22°C, 습도 45%', name='get_weather', call_id='call_2dXBsmW0gw7enfR37799JWtk', is_error=False)]
---------- TextMessage (debug_agent) ----------
오늘 서울의 날씨는 맑고 기온은 22°C이며, 습도는 45%입니다.


---
## 13. 통합 실습: 코드 리뷰 에이전트 팀

4개의 전문 Agent(Architect, SecurityReviewer, CodeOptimizer, Summarizer)가 협업하여 코드를 리뷰합니다.

### 13-1. 도구 정의

In [27]:
async def count_lines(code: str) -> str:
    """코드의 줄 수, 함수 수, 클래스 수를 분석합니다."""
    lines = code.strip().split("\n")
    func_count = sum(1 for line in lines if line.strip().startswith("def "))
    class_count = sum(1 for line in lines if line.strip().startswith("class "))
    return (
        f"분석 결과:\n"
        f"  - 총 줄 수: {len(lines)}\n"
        f"  - 함수 수: {func_count}\n"
        f"  - 클래스 수: {class_count}"
    )


async def check_security_patterns(code: str) -> str:
    """코드에서 보안 취약점 패턴을 검사합니다."""
    issues = []
    if "eval(" in code:
        issues.append("[심각] eval() 사용 감지 - 코드 인젝션 위험")
    if "exec(" in code:
        issues.append("[심각] exec() 사용 감지 - 코드 인젝션 위험")
    if "password" in code.lower() and "=" in code:
        issues.append("[심각] 하드코딩된 비밀번호 의심")
    if "SELECT" in code and "+" in code:
        issues.append("[심각] SQL 인젝션 가능성 - 문자열 연결로 쿼리 생성")
    if "import os" in code and "system(" in code:
        issues.append("[경고] os.system() 사용 - 명령어 인젝션 위험")
    if "pickle.load" in code:
        issues.append("[경고] pickle.load() 사용 - 역직렬화 공격 위험")

    if issues:
        return "보안 취약점 발견:\n" + "\n".join(f"  {i}" for i in issues)
    return "주요 보안 패턴 이슈 없음"


# 도구 테스트
test_code = """
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + str(user_id)
    result = eval(db.execute(query))
    password = "admin123"
    return result
"""

print(await count_lines(test_code))
print()
print(await check_security_patterns(test_code))

분석 결과:
  - 총 줄 수: 5
  - 함수 수: 1
  - 클래스 수: 0

보안 취약점 발견:
  [심각] eval() 사용 감지 - 코드 인젝션 위험
  [심각] 하드코딩된 비밀번호 의심
  [심각] SQL 인젝션 가능성 - 문자열 연결로 쿼리 생성


### 13-2. 코드 리뷰 Agent 팀 구성

In [28]:
architect = AssistantAgent(
    "Architect",
    description="코드의 구조, 설계 패턴, 아키텍처를 분석하는 에이전트.",
    model_client=model_client,
    tools=[count_lines],
    system_message="""코드의 구조적 품질을 분석하세요:
    - SOLID 원칙 준수 여부
    - 함수/클래스 설계
    - 네이밍 컨벤션
    - 관심사 분리(SoC)
    count_lines 도구로 코드 통계를 먼저 확인하세요.""",
)

security_reviewer = AssistantAgent(
    "SecurityReviewer",
    description="코드의 보안 취약점을 검토하는 에이전트.",
    model_client=model_client,
    tools=[check_security_patterns],
    system_message="""코드의 보안 취약점을 검토하세요:
    - check_security_patterns 도구로 자동 검사를 먼저 수행하세요
    - 입력 검증, 인젝션 위험, 인증/인가, 민감 데이터 노출을 확인하세요""",
)

optimizer = AssistantAgent(
    "CodeOptimizer",
    description="코드의 성능과 효율성을 분석하고 개선안을 제시하는 에이전트.",
    model_client=model_client,
    system_message="""코드의 성능 최적화 방안을 제시하세요:
    - 시간/공간 복잡도
    - 불필요한 연산 제거
    - 알고리즘 개선 기회""",
)

review_summarizer = AssistantAgent(
    "Summarizer",
    description="모든 리뷰 의견을 종합하여 최종 코드 리뷰를 작성하는 에이전트.",
    model_client=model_client,
    system_message="""다른 에이전트들의 리뷰를 종합하여 최종 코드 리뷰를 작성하세요.
    형식:
    ## 종합 코드 리뷰
    ### 심각도 상
    - ...
    ### 심각도 중
    - ...
    ### 심각도 하
    - ...
    ### 개선된 코드 제안
    최종 리뷰 작성 완료 시 반드시 'REVIEW_COMPLETE'라고 마지막에 적으세요.""",
)

print("Agent 4개 생성 완료: Architect, SecurityReviewer, CodeOptimizer, Summarizer")

Agent 4개 생성 완료: Architect, SecurityReviewer, CodeOptimizer, Summarizer


### 13-3. 코드 리뷰 실행

In [29]:
review_termination = TextMentionTermination("REVIEW_COMPLETE") | MaxMessageTermination(20)

review_team = SelectorGroupChat(
    [architect, security_reviewer, optimizer, review_summarizer],
    model_client=model_client,
    termination_condition=review_termination,
)

# 리뷰할 코드
code_to_review = """
def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + str(user_id)
    result = eval(db.execute(query))
    password = "admin123"
    return result

def process_data(data):
    output = []
    for i in range(len(data)):
        for j in range(len(data)):
            if data[i] == data[j] and i != j:
                output.append(data[i])
    return output
"""

result = await Console(review_team.run_stream(
    task=f"다음 Python 코드를 리뷰해주세요:\n```python\n{code_to_review}\n```"
))

---------- TextMessage (user) ----------
다음 Python 코드를 리뷰해주세요:
```python

def get_user(user_id):
    query = "SELECT * FROM users WHERE id = " + str(user_id)
    result = eval(db.execute(query))
    password = "admin123"
    return result

def process_data(data):
    output = []
    for i in range(len(data)):
        for j in range(len(data)):
            if data[i] == data[j] and i != j:
                output.append(data[i])
    return output

```
---------- ToolCallRequestEvent (SecurityReviewer) ----------
[FunctionCall(id='call_45GNef3ya8XcdsFLwqZqWiY6', arguments='{"code": "def get_user(user_id):\\n    query = \\"SELECT * FROM users WHERE id = \\" + str(user_id)\\n    result = eval(db.execute(query))\\n    password = \\"admin123\\"\\n    return result\\n\\ndef process_data(data):\\n    output = []\\n    for i in range(len(data)):\\n        for j in range(len(data)):\\n            if data[i] == data[j] and i != j:\\n                output.append(data[i])\\n    return output\\n"}',

In [30]:
# 코드 리뷰 결과 분석
print("=" * 60)
print("코드 리뷰 참여 Agent 분석")
print("=" * 60)

from collections import Counter
agent_counts = Counter(msg.source for msg in result.messages if msg.source != "user")
for agent_name, count in agent_counts.most_common():
    print(f"  {agent_name}: {count}회 발언")

print(f"\n총 메시지: {len(result.messages)}개")
print(f"종료 사유: {result.stop_reason}")

코드 리뷰 참여 Agent 분석
  SecurityReviewer: 3회 발언
  Architect: 3회 발언
  CodeOptimizer: 1회 발언
  Summarizer: 1회 발언

총 메시지: 9개
종료 사유: Text 'REVIEW_COMPLETE' mentioned


---
## 14. 실전 팁과 베스트 프랙티스

### 14-1. selector_func으로 커스텀 라우팅

SelectorGroupChat에서 `selector_func`을 사용하면 LLM 기반 선택을 보완하거나 오버라이드할 수 있습니다.

In [32]:
from typing import Sequence
from autogen_agentchat.messages import BaseAgentEvent, BaseChatMessage

# 커스텀 선택 함수: 항상 Planner를 먼저 거치도록 라우팅
def planner_first_selector(
    messages: Sequence[BaseAgentEvent | BaseChatMessage],
) -> str | None:
    """마지막 발언자가 Planner가 아니면 Planner로 보내고,
    Planner면 None(LLM 선택)으로 폴백"""
    if not messages:
        return "Planner"
    last_source = messages[-1].source
    if last_source == "Planner":
        return None  # LLM이 다음 Agent 선택
    return "Planner"  # 항상 Planner를 거침


planner_f = AssistantAgent(
    "Planner",
    description="작업을 계획하는 에이전트.",
    model_client=model_client,
    system_message="작업을 분석하고 다음 단계를 지시하세요. 완료 시 'COMPLETE'라고 적으세요.",
)
worker_f = AssistantAgent(
    "Worker",
    description="실제 작업을 수행하는 에이전트.",
    model_client=model_client,
    system_message="Planner의 지시에 따라 작업을 수행하세요.",
)

term_f = TextMentionTermination("COMPLETE") | MaxMessageTermination(8)
team_f = SelectorGroupChat(
    [planner_f, worker_f],
    model_client=model_client,
    termination_condition=term_f,
    selector_func=planner_first_selector,
)

result = await Console(team_f.run_stream(task="'Hello World'를 5개 언어로 번역해줘"))

# 발언 순서 확인 (항상 Planner를 거침)
print("\n발언 순서:")
for msg in result.messages:
    print(f"  → {msg.source}")

---------- TextMessage (user) ----------
'Hello World'를 5개 언어로 번역해줘
---------- TextMessage (Planner) ----------
Sure! Here is "Hello World" translated into five different languages:

1. Spanish: Hola Mundo
2. French: Bonjour le Monde
3. German: Hallo Welt
4. Italian: Ciao Mondo
5. Chinese (Simplified): 你好，世界

COMPLETE

발언 순서:
  → user
  → Planner


### 14-2. Writer-Critic Reflection 패턴 (RoundRobin)

가장 대표적인 RoundRobin 활용 패턴입니다. Writer가 글을 쓰고 Critic이 피드백하는 반복 구조.

In [33]:
# Writer-Critic Reflection 패턴
reflection_writer = AssistantAgent(
    "writer",
    model_client=model_client,
    system_message="""당신은 기술 블로그 작가입니다.
    주어진 주제로 짧은 글(200자 내외)을 작성하세요.
    Critic의 피드백이 있으면 반영하여 수정하세요.""",
)

reflection_critic = AssistantAgent(
    "critic",
    model_client=model_client,
    system_message="""당신은 기술 글 편집자입니다. 글을 검토하고 구체적인 피드백을 제공하세요.
    평가 기준: 정확성, 가독성, 구체성
    만족스러우면 'APPROVE'라고만 응답하세요.
    최대 2번까지만 피드백하고, 3번째에는 반드시 APPROVE하세요.""",
)

reflection_term = TextMentionTermination("APPROVE") | MaxMessageTermination(8)
reflection_team = RoundRobinGroupChat(
    [reflection_writer, reflection_critic],
    termination_condition=reflection_term,
)

result = await Console(reflection_team.run_stream(
    task="Python의 async/await가 필요한 이유를 설명하는 짧은 글을 써줘"
))

# 몇 라운드만에 완료되었는지 확인
rounds = sum(1 for msg in result.messages if msg.source == "writer")
print(f"\n작성 라운드: {rounds}회")
print(f"종료 사유: {result.stop_reason}")

---------- TextMessage (user) ----------
Python의 async/await가 필요한 이유를 설명하는 짧은 글을 써줘
---------- TextMessage (writer) ----------
Python의 `async/await` 구문은 비동기 프로그래밍을 가능하게 하여 응답성을 높이고, I/O 바운드 작업의 효율성을 극대화합니다. 전통적인 동기 프로그래밍에서는 각 작업이 종료될 때까지 대기해야 하므로, 시간이 오래 걸리는 작업이 전체 응답성을 저하시킬 수 있습니다. `async/await`를 사용하면 비동기 함수들이 서로 독립적으로 실행되며, 특정 작업이 완료되기를 기다리지 않고 다른 작업을 계속 수행할 수 있어 자원의 활용도가 올라갑니다. 특히 웹 서버나 네트워크 요청 처리와 같이 많은 I/O 대기 시간이 소요되는 경우에 이점을 제공합니다.
---------- TextMessage (critic) ----------
1. "Python의 `async/await` 구문은 비동기 프로그래밍을 가능하게 하여" 부분에서 오해의 소지가 있을 수 있습니다. Python에서는 `async/await` 외에도 다양한 방법으로 비동기 프로그래밍이 가능합니다. 예를 들어, `concurrent.futures`와 같은 모듈도 사용할 수 있습니다. 따라서, "Python에서는 `async/await`를 통해 더 간결하고 직관적인 비동기 프로그래밍을 지원합니다."라고 표현하는 것이 더 명확할 것입니다.

2. "응답성을 높이고, I/O 바운드 작업의 효율성을 극대화합니다."는 개선의 여지가 있습니다. 비동기 프로그램은 CPU 바운드 작업보다는 I/O 바운드 작업에서 특히 더 많은 이득을 볼 수 있다는 점을 강조하면 좋겠습니다. 예를 들어, "응답성 향상과 함께 I/O 바운드 작업에서의 효율성을 극대화합니다."라고 말할 수 있습니다.

3. "전통적인 동기 프로그래밍에서는 각 작업이 종료될 때까지 대기해야 하므로, 시간이 오래 걸리는 작업이 전체

### 14-3. 전체 패턴 비교 실험

동일한 작업을 RoundRobin, SelectorGroupChat, Swarm 세 패턴으로 실행하고 결과를 비교합니다.

In [34]:
import time

task = "Python과 JavaScript의 장단점을 비교해줘"

# === 패턴 1: RoundRobin ===
rr_a = AssistantAgent("analyst", model_client=model_client,
    system_message="기술 비교 분석을 수행하세요.")
rr_b = AssistantAgent("reviewer", model_client=model_client,
    system_message="분석을 검토하세요. 만족스러우면 'DONE'이라고 응답하세요.")
rr_term = TextMentionTermination("DONE") | MaxMessageTermination(6)
rr_team = RoundRobinGroupChat([rr_a, rr_b], termination_condition=rr_term)

t0 = time.time()
rr_result = await rr_team.run(task=task)
rr_time = time.time() - t0

# === 패턴 2: SelectorGroupChat ===
sg_a = AssistantAgent("PythonExpert", description="Python 전문가.",
    model_client=model_client, system_message="Python 관점에서 분석하세요.")
sg_b = AssistantAgent("JSExpert", description="JavaScript 전문가.",
    model_client=model_client, system_message="JavaScript 관점에서 분석하세요.")
sg_c = AssistantAgent("Summarizer", description="결과를 종합하는 에이전트.",
    model_client=model_client,
    system_message="두 전문가의 의견을 종합하세요. 완료 시 'DONE'이라고 적으세요.")
sg_term = TextMentionTermination("DONE") | MaxMessageTermination(8)
sg_team = SelectorGroupChat([sg_a, sg_b, sg_c], model_client=model_client,
    termination_condition=sg_term)

t0 = time.time()
sg_result = await sg_team.run(task=task)
sg_time = time.time() - t0

# === 결과 비교 ===
print("=" * 60)
print("패턴별 결과 비교")
print("=" * 60)
print(f"{'패턴':<25} {'메시지 수':>8} {'소요 시간':>10}")
print("-" * 60)
print(f"{'RoundRobinGroupChat':<25} {len(rr_result.messages):>8} {rr_time:>9.1f}s")
print(f"{'SelectorGroupChat':<25} {len(sg_result.messages):>8} {sg_time:>9.1f}s")

패턴별 결과 비교
패턴                           메시지 수      소요 시간
------------------------------------------------------------
RoundRobinGroupChat              3      14.1s
SelectorGroupChat                4      25.2s


### 14-4. 정리: 패턴 선택 가이드

| 패턴 | 적합한 작업 | 복잡도 |
|:---:|:---:|:---:|
| **RoundRobin** | Writer-Critic, 순차 피드백 | 낮음 |
| **SelectorGroupChat** | 전문가 협업, 동적 라우팅 | 중간 |
| **Swarm** | 워크플로우, 사용자 핸드오프 | 높음 |

In [35]:
# 임시 파일 정리
import os
if os.path.exists("team_state.json"):
    os.remove("team_state.json")
    print("team_state.json 정리 완료")

team_state.json 정리 완료
